### makemore: becoming a backprop ninja
![image](../images/swole-doge-minimal-template-pair.png)

In [98]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random
%matplotlib inline

In [99]:
words = open('src/names.txt', 'r').read().splitlines()
print(words[:8], len(words))

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia'] 32033


In [100]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi  = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos  = {i: s for s, i in stoi.items()}
vocab_size = len(itos)

In [101]:
block_size = 3

def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr,  Ytr  = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte,  Yte  = build_dataset(words[n2:])

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


til now, none of these ever changed

---

In [102]:
def cmp(s, dt, t):
    """Utility function comparing manual and pytorch gradient.
    Args: s-"""
    ex      = torch.all(dt == t.grad).item()
    app     = torch.allclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    print(f'{s:15s}| exact: {str(ex):5s}| approximate: {str(app):5s}| maxdiff: {str(maxdiff):5s}')

In [103]:
n_embd = 10
n_hidd = 64

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((vocab_size, n_embd),           generator = g)

# Layer1
W1 = torch.randn((block_size * n_embd, n_hidd), generator = g) * ((5 / 3) / ((block_size * n_embd) ** 0.5))# 0.2
b1 = torch.randn(n_hidd,                        generator = g) * 0.1

# Layer2
W2 = torch.randn((n_hidd, vocab_size),          generator = g) * 0.1
b2 = torch.randn(vocab_size,                    generator = g) * 0.1

# Batch Normalization parameters
bngain         = torch.ones((1, n_hidd)) * 0.1 + 1.0
bnbias         = torch.zeros((1, n_hidd)) * 0.1
# bnstd_running  = torch.ones((1, n_hidd))
# bnmean_running = torch.zeros((1, n_hidd))

parameters = [C, W1, b1, W2, b2, bnbias, bngain]
print(sum(p.nelement() for p in parameters))
for p in parameters:
    p.requires_grad = True

4137


In [104]:
# Construct minibatches
batch_size = 32
n          = batch_size # For convenience
ix         = torch.randint(0, Xtr.shape[0], (batch_size, ), generator = g)
Xb, Yb     = Xtr[ix], Ytr[ix]

In [105]:
# forward pass, "chunkated" into smaller steps that are possible to backward once at a time
emb    = C[Xb]
embcat = emb.view(emb.shape[0], -1)

# Linear layer 1
hprebn    = embcat @ W1 + b1
bnmeani   = 1 / n * hprebn.sum(0, keepdim=True)
bndiff    = hprebn - bnmeani
bndiff2   = bndiff ** 2
bnvar     = 1 / (n - 1) * (bndiff2).sum(0, keepdim=True) # note: Bessel's correction
bnvar_inv = (bnvar + 1e-5) ** -0.5
bnraw     = bndiff * bnvar_inv
hpreact   = bngain * bnraw + bnbias
bnstdi    = hpreact.std(0, keepdim=True)


# Non-Linearity
h = torch.tanh(hpreact)

# Linear layer 2
logits = h @ W2 + b2

# Cross-entropy loss
logits_maxes   = logits.max(1, keepdim=True).values
norm_logits    = logits - logits_maxes # subtract max for numerical stability
counts         = norm_logits.exp()
counts_sum     = counts.sum(1, keepdim=True)
counts_sum_inv = counts_sum ** -1
probs          = counts * counts_sum_inv
logprobs       = probs.log()
loss           = -logprobs[range(n), Yb].mean()

# backward pass
for p in parameters:
    p.grad = None

# What a piece of ugly code.
for t in [emb, embcat,
          hprebn, bnmeani, bndiff, bndiff2, bnvar, bnvar_inv, bnraw, bnstdi, hpreact,
          h,
          logits,
          logits_maxes, norm_logits, counts, counts_sum, counts_sum_inv, probs, logprobs]:
    t.retain_grad()
loss.backward()
loss

tensor(3.3482, grad_fn=<NegBackward0>)

In [106]:
# Exercise 1: backprop through the whole thing manually,
# backpropagating through exactly all of the variables
# as they are defined in the forward pass above, one by one

# -----------------
# YOUR CODE HERE :)
# Codes below are quite messy,
# I would say initialize derivatives to be zero then use +=s is recommended.
dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n), Yb] = -1 / n

dprobs = (probs ** -1) * dlogprobs
dcounts_sum_inv = (counts * dprobs).sum(1, keepdim = True)
dcounts_sum = (-counts_sum ** -2) * dcounts_sum_inv
dcounts = torch.ones_like(counts) * dcounts_sum + counts_sum_inv * dprobs
dnorm_logits = norm_logits.exp() * dcounts
dlogits_maxes = -1 * dnorm_logits.sum(1, keepdim = True) # -1 * dnorm_logits

indices = logits.max(1, keepdim=True).indices.sum(1).tolist()
delta = torch.zeros_like(dnorm_logits)
delta[range(n), indices] = 1
delta = delta * dlogits_maxes
dlogits = torch.ones_like(dnorm_logits) * dnorm_logits
dlogits = dlogits + delta

dh = dlogits @ W2.T
dW2 = h.T @ dlogits
db2 = dlogits.sum(0)

dhpreact = (1 - h ** 2) * dh

dbngain = (bnraw * dhpreact).sum(0, keepdim = True)
dbnbias = dhpreact.sum(0, keepdim = True)
dbnraw = (bngain * dhpreact)
dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim = True)
dbnvar = (-0.5 * (bnvar + 1e-5) ** -1.5) * dbnvar_inv
dbndiff2 = torch.ones_like(bndiff2) / (n - 1) * dbnvar
dbndiff = 2 * bndiff * dbndiff2 + bnvar_inv * dbnraw
dbnmeani = (-1 * torch.ones_like(bnmeani) * dbndiff).sum(0, keepdim = True)
dhprebn = 1 / n * torch.ones_like(hprebn) * dbnmeani + dbndiff

dembcat = dhprebn @ W1.T
dW1 = embcat.T @ dhprebn
db1 = dhprebn.sum(0)

demb = dembcat.view(emb.shape[0], emb.shape[1], -1)

dC = torch.zeros_like(C)
for k in range(Xb.shape[0]):
    for j in range(Xb.shape[1]):
        ix = Xb[k,j]
        dC[ix] += demb[k,j]
# -----------------

cmp('logprobs', dlogprobs, logprobs)
cmp('probs', dprobs, probs)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)
cmp('counts_sum', dcounts_sum, counts_sum)
cmp('counts', dcounts, counts)
cmp('norm_logits', dnorm_logits, norm_logits)
cmp('logit_maxes', dlogits_maxes, logits_maxes)
cmp('logits', dlogits, logits)
cmp('h', dh, h)
cmp('W2', dW2, W2)
cmp('b2', db2, b2)
cmp('hpreact', dhpreact, hpreact)
cmp('bngain', dbngain, bngain)
cmp('bnbias', dbnbias, bnbias)
cmp('bnraw', dbnraw, bnraw)
cmp('bnvar_inv', dbnvar_inv, bnvar_inv)
cmp('bnvar', dbnvar, bnvar)
cmp('bndiff2', dbndiff2, bndiff2)
cmp('bndiff', dbndiff, bndiff)
cmp('bnmeani', dbnmeani, bnmeani)
cmp('hprebn', dhprebn, hprebn)
cmp('embcat', dembcat, embcat)
cmp('W1', dW1, W1)
cmp('b1', db1, b1)
cmp('emb', demb, emb)
cmp('C', dC, C)

logprobs       | exact: True | approximate: True | maxdiff: 0.0  
probs          | exact: True | approximate: True | maxdiff: 0.0  
counts_sum_inv | exact: True | approximate: True | maxdiff: 0.0  
counts_sum     | exact: True | approximate: True | maxdiff: 0.0  
counts         | exact: True | approximate: True | maxdiff: 0.0  
norm_logits    | exact: True | approximate: True | maxdiff: 0.0  
logit_maxes    | exact: True | approximate: True | maxdiff: 0.0  
logits         | exact: True | approximate: True | maxdiff: 0.0  
h              | exact: True | approximate: True | maxdiff: 0.0  
W2             | exact: True | approximate: True | maxdiff: 0.0  
b2             | exact: True | approximate: True | maxdiff: 0.0  
hpreact        | exact: True | approximate: True | maxdiff: 0.0  
bngain         | exact: True | approximate: True | maxdiff: 0.0  
bnbias         | exact: True | approximate: True | maxdiff: 0.0  
bnraw          | exact: True | approximate: True | maxdiff: 0.0  
bnvar_inv 

In [107]:
# Exercise 2: backprop through cross_entropy but all in one go
# to complete this challenge look at the mathematical expression of the loss,
# take the derivative, simplify the expression, and just write it out

# forward pass

# before:
# logit_maxes = logits.max(1, keepdim=True).values
# norm_logits = logits - logit_maxes # subtract max for numerical stability
# counts = norm_logits.exp()
# counts_sum = counts.sum(1, keepdims=True)
# counts_sum_inv = counts_sum**-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
# probs = counts * counts_sum_inv
# logprobs = probs.log()
# loss = -logprobs[range(n), Yb].mean()

# now:
loss_fast = F.cross_entropy(logits, Yb)
print(loss_fast.item(), 'diff:', (loss_fast - loss).item())

3.348198175430298 diff: 0.0


In [108]:
# backward pass


# -----------------
# YOUR CODE HERE :)
dlogits = F.softmax(logits, 1) # TODO. my solution is 3 lines
dlogits[range(n), Yb] -= 1
dlogits /= n
# -----------------

cmp('logits', dlogits, logits) # I can only get approximate to be true, my maxdiff is 6e-9

logits         | exact: False| approximate: True | maxdiff: 5.587935447692871e-09


In [109]:
# Exercise 3: backprop through batchnorm but all in one go
# to complete this challenge look at the mathematical expression of the output of batchnorm,
# take the derivative w.r.t. its input, simplify the expression, and just write it out
# BatchNorm paper: https://arxiv.org/abs/1502.03167

# forward pass

# before:
# bnmeani = 1/n*hprebn.sum(0, keepdim=True)
# bndiff = hprebn - bnmeani
# bndiff2 = bndiff**2
# bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n-1, not n)
# bnvar_inv = (bnvar + 1e-5)**-0.5
# bnraw = bndiff * bnvar_inv
# hpreact = bngain * bnraw + bnbias

# now:
hpreact_fast = bngain * (hprebn - hprebn.mean(0, keepdim=True)) / torch.sqrt(hprebn.var(0, keepdim=True, unbiased=True) + 1e-5) + bnbias
print('max diff:', (hpreact_fast - hpreact).abs().max())

max diff: tensor(7.1526e-07, grad_fn=<MaxBackward1>)


In [110]:
# backward pass

# before we had:
# dbnraw = bngain * dhpreact
# dbndiff = bnvar_inv * dbnraw
# dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)
# dbnvar = (-0.5*(bnvar + 1e-5)**-1.5) * dbnvar_inv
# dbndiff2 = (1.0/(n-1))*torch.ones_like(bndiff2) * dbnvar
# dbndiff += (2*bndiff) * dbndiff2
# dhprebn = dbndiff.clone()
# dbnmeani = (-dbndiff).sum(0)
# dhprebn += 1.0/n * (torch.ones_like(hprebn) * dbnmeani)

# calculate dhprebn given dhpreact (i.e. backprop through the batchnorm)
# (you'll also need to use some of the variables from the forward pass up above)

# -----------------
# YOUR CODE HERE :)
# dhprebn = (dlogits @ W2.T) * (1 - h ** 2) * (bngain * (bnvar_inv - 1 / n * bnvar_inv ** 3 * bndiff)) # TODO. my solution is 1 (long) line

# I ignored bessel correction and got the wrong answer.
dhprebn = bngain*bnvar_inv/n * (n*dhpreact - dhpreact.sum(0) - n/(n-1)*bnraw*(dhpreact*bnraw).sum(0))
# -----------------

cmp('hprebn', dhprebn, hprebn) # I can only get approximate to be true, my maxdiff is 9e-10

hprebn         | exact: False| approximate: True | maxdiff: 9.313225746154785e-10


In [113]:
# Exercise 4: putting it all together!
# Train the MLP neural net with your own backward pass

# init
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 200 # the number of neurons in the hidden layer of the MLP

g = torch.Generator().manual_seed(2147483647) # for reproducibility
C  = torch.randn((vocab_size, n_embd),            generator=g)
# Layer 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn(n_hidden,                        generator=g) * 0.1
# Layer 2
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.1
b2 = torch.randn(vocab_size,                      generator=g) * 0.1
# BatchNorm parameters
bngain = torch.randn((1, n_hidden))*0.1 + 1.0
bnbias = torch.randn((1, n_hidden))*0.1

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True

# same optimization as last time
max_steps = 200000
batch_size = 32
n = batch_size # convenience
lossi = []

# use this context manager for efficiency once your backward pass is written (TODO)
#with torch.no_grad():

# kick off optimization
for i in range(max_steps):

    # minibatch construct
    ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
    Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y

    # forward pass
    emb = C[Xb] # embed the characters into vectors
    embcat = emb.view(emb.shape[0], -1) # concatenate the vectors
    # Linear layer
    hprebn = embcat @ W1 + b1 # hidden layer pre-activation
    # BatchNorm layer
    # -------------------------------------------------------------
    bnmean = hprebn.mean(0, keepdim=True)
    bnvar = hprebn.var(0, keepdim=True, unbiased=True)
    bnvar_inv = (bnvar + 1e-5)**-0.5
    bnraw = (hprebn - bnmean) * bnvar_inv
    hpreact = bngain * bnraw + bnbias
    # -------------------------------------------------------------
    # Non-linearity
    h = torch.tanh(hpreact) # hidden layer
    logits = h @ W2 + b2 # output layer
    loss = F.cross_entropy(logits, Yb) # loss function

    # backward pass
    for p in parameters:
      p.grad = None
    # loss.backward() # use this for correctness comparisons, delete it later!

    # manual backprop! #swole_doge_meme
    # -----------------
    # YOUR CODE HERE :)
    # dC, dW1, db1, dW2, db2, dbngain, dbnbias = None, None, None, None, None, None, None
    dlogits = F.softmax(logits, 1) # TODO. my solution is 3 lines
    dlogits[range(n), Yb] -= 1
    dlogits /= n

    dh = dlogits @ W2.T
    dW2 = h.T @ dlogits
    db2 = dlogits.sum(0)

    dhpreact = (1 - h ** 2) * dh

    dbngain = (bnraw * dhpreact).sum(0, keepdim=True)
    dbnbias = dhpreact.sum(0, keepdim=True)
    dhprebn = bngain*bnvar_inv/n * (n*dhpreact - dhpreact.sum(0) - n/(n-1)*bnraw*(dhpreact*bnraw).sum(0))

    dembcat = dhprebn @ W1.T
    dW1 = embcat.T @ dhprebn
    db1 = dhprebn.sum(0)

    demb = dembcat.view(emb.shape[0], emb.shape[1], -1)

    dC = torch.zeros_like(C)
    for k in range(Xb.shape[0]):
        for j in range(Xb.shape[1]):
            ix = Xb[k,j]
            dC[ix] += demb[k,j]
    grads = [dC, dW1, db1, dW2, db2, dbngain, dbnbias]
    # -----------------

    # update
    lr = 0.1 if i < 100000 else 0.01 # step learning rate decay
    for p, grad in zip(parameters, grads):
        # p.data += -lr * p.grad # old way of cheems doge (using PyTorch grad from .backward())
        p.data += -lr * grad # new way of swole doge TODO: enable

    # track stats
    if i % 10000 == 0: # print every once in a while
        print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
    lossi.append(loss.log10().item())

    # if i >= 100: # TODO: delete early breaking when you're ready to train the full net
    #     break

12297
      0/ 200000: 3.8138
  10000/ 200000: 2.1667
  20000/ 200000: 2.3677
  30000/ 200000: 2.4765
  40000/ 200000: 1.9489
  50000/ 200000: 2.3905
  60000/ 200000: 2.4371
  70000/ 200000: 2.0539
  80000/ 200000: 2.3094
  90000/ 200000: 2.0749
 100000/ 200000: 1.9902
 110000/ 200000: 2.2402
 120000/ 200000: 2.0004
 130000/ 200000: 2.3871
 140000/ 200000: 2.3275
 150000/ 200000: 2.2115
 160000/ 200000: 1.9775
 170000/ 200000: 1.7612
 180000/ 200000: 1.9729
 190000/ 200000: 1.8864


In [114]:
# useful for checking your gradients
# for p,g in zip(parameters, grads):
#     cmp(str(tuple(p.shape)), g, p)
loss.item()

2.3608202934265137

In [115]:
# calibrate the batch norm at the end of training

with torch.no_grad():
  # pass the training set through
  emb = C[Xtr]
  embcat = emb.view(emb.shape[0], -1)
  hpreact = embcat @ W1 + b1
  # measure the mean/std over the entire training set
  bnmean = hpreact.mean(0, keepdim=True)
  bnvar = hpreact.var(0, keepdim=True, unbiased=True)


In [116]:
# evaluate train and val loss

@torch.no_grad() # this decorator disables gradient tracking
def split_loss(split):
  x,y = {
    'train': (Xtr, Ytr),
    'val': (Xdev, Ydev),
    'test': (Xte, Yte),
  }[split]
  emb = C[x] # (N, block_size, n_embd)
  embcat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
  hpreact = embcat @ W1 + b1
  hpreact = bngain * (hpreact - bnmean) * (bnvar + 1e-5)**-0.5 + bnbias
  h = torch.tanh(hpreact) # (N, n_hidden)
  logits = h @ W2 + b2 # (N, vocab_size)
  loss = F.cross_entropy(logits, y)
  print(split, loss.item())

split_loss('train')
split_loss('val')

train 2.0712552070617676
val 2.108290672302246


In [117]:
# sample from the model
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):

    out = []
    context = [0] * block_size # initialize with all ...
    while True:
      # forward pass
      emb = C[torch.tensor([context])] # (1,block_size,d)
      embcat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
      hpreact = embcat @ W1 + b1
      hpreact = bngain * (hpreact - bnmean) * (bnvar + 1e-5)**-0.5 + bnbias
      h = torch.tanh(hpreact) # (N, n_hidden)
      logits = h @ W2 + b2 # (N, vocab_size)
      # sample
      probs = F.softmax(logits, dim=1)
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break

    print(''.join(itos[i] for i in out))

carmahzaul.
harli.
jari.
reity.
skaelane.
mahnen.
den.
arcie.
qui.
nellara.
chaiha.
kaleigh.
ham.
joce.
quint.
shon.
marianni.
waythoniearynix.
kaellinsley.
daedi.
